In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.LM(
    params=model.parameters(),
    model=model,
    lr=1,
    mu=0.001,
    mu_dec=0.1,
    fletcher=False,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    solver="solve",
    batch_size=100,
    # batch_size=None,
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.05467768386006355
epoch:  1, loss: 0.039154618978500366
epoch:  2, loss: 0.0333964079618454
epoch:  3, loss: 0.032438699156045914
epoch:  4, loss: 0.031731266528367996
epoch:  5, loss: 0.03055332787334919
epoch:  6, loss: 0.02975081466138363
epoch:  7, loss: 0.02033267915248871
epoch:  8, loss: 0.018374446779489517
epoch:  9, loss: 0.01704799383878708
epoch:  10, loss: 0.01619364134967327
epoch:  11, loss: 0.015646593645215034
epoch:  12, loss: 0.01542879082262516
epoch:  13, loss: 0.014555787667632103
epoch:  14, loss: 0.014040403068065643
epoch:  15, loss: 0.01372923981398344
epoch:  16, loss: 0.013379323296248913
epoch:  17, loss: 0.012755476869642735
epoch:  18, loss: 0.012422394938766956
epoch:  19, loss: 0.012226236052811146
epoch:  20, loss: 0.011805567890405655
epoch:  21, loss: 0.011387932114303112
epoch:  22, loss: 0.011179720051586628
epoch:  23, loss: 0.011153516359627247
epoch:  24, loss: 0.010601467452943325
epoch:  25, loss: 0.010368055664002895
epoch:

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9702084659950748
Test metrics:  R2 = 0.9564981724661432


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.LM(
    params=model.parameters(),
    model=model,
    lr=1,
    mu=0.001,
    mu_dec=0.1,
    fletcher=True,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    solver="pinv",
    batch_size=100,
    # batch_size=None,
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0

/home/eugeniolr/Documents/Eugenio/torch_numopt/src/torch_numopt/utils.py:44: UserWarning: torch.linalg.svd: During SVD computation with the selected cusolver driver, batches 0 failed to converge. A more accurate method will be used to compute the SVD as a fallback. Check doc at https://pytorch.org/docs/stable/generated/torch.linalg.svd.html (Triggered internally at /pytorch/aten/src/ATen/native/cuda/linalg/BatchLinearAlgebraLib.cpp:690.)
  U, S, Vt = torch.linalg.svd(mat)


, loss: 0.4431418180465698
epoch:  1, loss: 0.31882742047309875
epoch:  2, loss: 0.22776153683662415
epoch:  3, loss: 0.1469728648662567
epoch:  4, loss: 0.08189981430768967
epoch:  5, loss: 0.048097141087055206
epoch:  6, loss: 0.0345391109585762
epoch:  7, loss: 0.03334805741906166
epoch:  8, loss: 0.0233281459659338
epoch:  9, loss: 0.02067815326154232
epoch:  10, loss: 0.0186286810785532
epoch:  11, loss: 0.011997757479548454
epoch:  12, loss: 0.010740331374108791
epoch:  13, loss: 0.009341905824840069
epoch:  14, loss: 0.008783902041614056
epoch:  15, loss: 0.008536046370863914
epoch:  16, loss: 0.007982736453413963
epoch:  17, loss: 0.007712867576628923
epoch:  18, loss: 0.007629432715475559
epoch:  19, loss: 0.00754024600610137
epoch:  20, loss: 0.007448100484907627
epoch:  21, loss: 0.007411138154566288
epoch:  22, loss: 0.0073910728096961975
epoch:  23, loss: 0.0073793684132397175
epoch:  24, loss: 0.00736993970349431
epoch:  25, loss: 0.0073535083793103695
epoch:  26, loss: 0

In [9]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7086681734630786
Test metrics:  R2 = 0.6902413934940552
